In [1]:
from passwords import *
from pprint import pprint
import pandas as pd
import pyodbc
import os
import boto3
import datetime as dt

In [2]:
print(f'Latest run date: {dt.datetime.today()}')

Latest run date: 2024-03-20 12:41:19.081723


### Functions

In [3]:
# upload to s3
def upload_to_s3(aws_access_key_id, aws_secret_access_key, str_local_path, str_bucket_key, str_bucket_name):
    # init client
    cls_client = boto3.client(
        's3',
        aws_access_key_id=aws_access_key_id,
        aws_secret_access_key=aws_secret_access_key,
        aws_session_token=None,
    )
    # upload
    cls_client.upload_file(
        str_local_path, 
        str_bucket_name, 
        str_bucket_key,
    )

### Constants

In [4]:
# project
str_project = os.getcwd().split('\\')[4].replace('_','-')
print(f'Project: {str_project}')
# task
str_task = os.getcwd().split('\\')[5]
print(f'Task: {str_task}')
# sub task
str_subtask = os.getcwd().split('\\')[6]
print(f'Subtask: {str_subtask}')
# output
str_dirname_output = './output'

Project: 20231010-gen-xii
Task: 14_loss_forecasting
Subtask: 01_pull_data


### Output directory

In [5]:
try:
    os.mkdir(str_dirname_output)
except FileExistsError:
    pass

### Read query

In [6]:
str_filepath = './sql/query_normal.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT\n'
 '\tbigAccountId,\n'
 '\tdtmBooking as dtmFunded,\n'
 '\tfltNetChgOff,\n'
 '\tMonthEndDate\n'
 'FROM riskdb.accountingReports.tblAccounting_LoanCOandNA_ME')


### Write into df

In [7]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# make col
df['bktype'] = 'nobk'

# show
df

Wall time: 30 s


,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype
0,370217,2006-08-30 17:12:22.000,6796.08,2019-12-31,nobk
1,306072,2005-10-11 17:27:40.000,17702.55,2019-12-31,nobk
2,245270,2004-10-13 00:00:00.000,4755.88,2019-12-31,nobk
3,196070,2002-07-29 00:00:00.000,9161.37,2019-12-31,nobk
4,239071,2004-08-26 00:00:00.000,15970.30,2019-12-31,nobk
...,...,...,...,...,...
5077702,2809081,2016-10-21 15:38:36.000,17408.06,2024-02-29,nobk
5077703,1604746,2014-05-29 11:26:45.000,2990.02,2024-02-29,nobk
5077704,375192,2006-09-21 15:24:52.000,4903.67,2024-02-29,nobk
5077705,1876609,2014-12-26 16:02:56.000,7724.09,2024-02-29,nobk


### Read query

In [8]:
str_filepath = './sql/query_bk.sql'
str_query = open(str_filepath, 'r').read()
pprint(str_query)

('SELECT\n'
 '\tbigAccountId,\n'
 '\tdtmFunded,\n'
 '\tmnyNetGainLoss as fltNetChgOff,\n'
 '\tdtmRunDate as MonthEndDate\n'
 'FROM riskdb.accountingReports.tblAccounting_ReportV11_ME')


### Write into df

In [9]:
%%time

# create connection
conn = pyodbc.connect(
    'Driver={SQL Server};'
    'Server=electra;'
    'Database=pfsdb;'
    'Trusted_Connection=yes;'
)
df_tmp = pd.read_sql_query(
    str_query, 
    con=conn,
)
# close
conn.close()

# make col
df_tmp['bktype'] = 'bk'

# show
df_tmp

Wall time: 511 ms


,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype
0,1427393,2013-12-30 17:26:10.000,0.00,2019-12-31,bk
1,998239,2012-09-28 16:20:09.000,16.00,2019-12-31,bk
2,1234987,2013-08-07 15:01:13.000,22.69,2019-12-31,bk
3,854538,2012-01-27 14:26:10.000,425.00,2019-12-31,bk
4,1670841,2014-07-22 15:44:21.000,112.27,2019-12-31,bk
...,...,...,...,...,...
122541,6928982,2023-07-07 14:36:42.827,0.00,2024-02-29,bk
122542,3802128,2018-08-24 16:52:43.000,533.49,2024-02-29,bk
122543,4446512,2019-07-02 15:06:10.000,0.00,2024-02-29,bk
122544,4049324,2019-01-03 10:13:10.000,53.54,2024-02-29,bk


### Concatenate

In [10]:
df = pd.concat([df, df_tmp])
del df_tmp
# show
df

,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype
0,370217,2006-08-30 17:12:22.000,6796.08,2019-12-31,nobk
1,306072,2005-10-11 17:27:40.000,17702.55,2019-12-31,nobk
2,245270,2004-10-13 00:00:00.000,4755.88,2019-12-31,nobk
3,196070,2002-07-29 00:00:00.000,9161.37,2019-12-31,nobk
4,239071,2004-08-26 00:00:00.000,15970.30,2019-12-31,nobk
...,...,...,...,...,...
122541,6928982,2023-07-07 14:36:42.827,0.00,2024-02-29 00:00:00,bk
122542,3802128,2018-08-24 16:52:43.000,533.49,2024-02-29 00:00:00,bk
122543,4446512,2019-07-02 15:06:10.000,0.00,2024-02-29 00:00:00,bk
122544,4049324,2019-01-03 10:13:10.000,53.54,2024-02-29 00:00:00,bk


### Convert funding month and month end date to first of each month

In [11]:
df['dtmFunded_first'] = df['dtmFunded'].dt.to_period('M').dt.to_timestamp()
df['MonthEndDate'] = pd.to_datetime(df['MonthEndDate'])
df['MonthEndDate_first'] = df['MonthEndDate'].dt.to_period('M').dt.to_timestamp()
df

,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype,dtmFunded_first,MonthEndDate_first
0,370217,2006-08-30 17:12:22.000,6796.08,2019-12-31,nobk,2006-08-01,2019-12-01
1,306072,2005-10-11 17:27:40.000,17702.55,2019-12-31,nobk,2005-10-01,2019-12-01
2,245270,2004-10-13 00:00:00.000,4755.88,2019-12-31,nobk,2004-10-01,2019-12-01
3,196070,2002-07-29 00:00:00.000,9161.37,2019-12-31,nobk,2002-07-01,2019-12-01
4,239071,2004-08-26 00:00:00.000,15970.30,2019-12-31,nobk,2004-08-01,2019-12-01
...,...,...,...,...,...,...,...
122541,6928982,2023-07-07 14:36:42.827,0.00,2024-02-29,bk,2023-07-01,2024-02-01
122542,3802128,2018-08-24 16:52:43.000,533.49,2024-02-29,bk,2018-08-01,2024-02-01
122543,4446512,2019-07-02 15:06:10.000,0.00,2024-02-29,bk,2019-07-01,2024-02-01
122544,4049324,2019-01-03 10:13:10.000,53.54,2024-02-29,bk,2019-01-01,2024-02-01


### Get months on books

In [12]:
df['years'] = df['MonthEndDate_first'].dt.year - df['dtmFunded_first'].dt.year
# convert to months
df['months'] = df['years'] * 12
# get difference in months
df['months_tmp'] = df['MonthEndDate_first'].dt.month - df['dtmFunded_first'].dt.month
# get mob
df['months_on_books'] = df['months'] + df['months_tmp']
# show
df

,bigAccountId,dtmFunded,fltNetChgOff,MonthEndDate,bktype,dtmFunded_first,MonthEndDate_first,years,months,months_tmp,months_on_books
0,370217,2006-08-30 17:12:22.000,6796.08,2019-12-31,nobk,2006-08-01,2019-12-01,13,156,4,160
1,306072,2005-10-11 17:27:40.000,17702.55,2019-12-31,nobk,2005-10-01,2019-12-01,14,168,2,170
2,245270,2004-10-13 00:00:00.000,4755.88,2019-12-31,nobk,2004-10-01,2019-12-01,15,180,2,182
3,196070,2002-07-29 00:00:00.000,9161.37,2019-12-31,nobk,2002-07-01,2019-12-01,17,204,5,209
4,239071,2004-08-26 00:00:00.000,15970.30,2019-12-31,nobk,2004-08-01,2019-12-01,15,180,4,184
...,...,...,...,...,...,...,...,...,...,...,...
122541,6928982,2023-07-07 14:36:42.827,0.00,2024-02-29,bk,2023-07-01,2024-02-01,1,12,-5,7
122542,3802128,2018-08-24 16:52:43.000,533.49,2024-02-29,bk,2018-08-01,2024-02-01,6,72,-6,66
122543,4446512,2019-07-02 15:06:10.000,0.00,2024-02-29,bk,2019-07-01,2024-02-01,5,60,-5,55
122544,4049324,2019-01-03 10:13:10.000,53.54,2024-02-29,bk,2019-01-01,2024-02-01,5,60,1,61


### Save as parquet

In [13]:
%%time

# save
str_filename = 'df_loss.gzip'
str_local_path = f'{str_dirname_output}/{str_filename}'
df.to_parquet(str_local_path, compression='gzip')

Wall time: 12 s


### Upload to s3

In [14]:
%%time

# upload
upload_to_s3(
    aws_access_key_id=AWS_ACCESS_KEY_ID, 
    aws_secret_access_key=AWS_SECRET_ACCESS_KEY, 
    str_local_path=str_local_path, 
    str_bucket_key=f'{str_task}/{str_subtask}/{str_filename}', 
    str_bucket_name=str_project,
)

Wall time: 2.23 s


### Clean-up

In [15]:
os.remove(str_local_path)